# Main Cleaning & Feature Engineering

***Round 1 (notebooks 01–02) resolved structural problems in the raw data and produced `games_clean_r1.csv`, 113,065 rows, 35 columns, non-game entries removed. Round 2 EDA (notebook 03) then identified the target variable and the feature shortlist that will enter modelling. This notebook applies those decisions and performs complete feature engineering before exporting final modelling-ready datasets.***

***The target `success_score = log1p(review_count) × wilson_lb` is constructed first, then zero-review games are excluded. Feature transformations follow: `log1p` on `game_age_days`, `Price`, `Achievements`, and `DLC count`; `is_free` derived as a binary flag; `num_tags` carried as-is; and `primary_genre` one-hot encoded against the top 20 genres. The dataset is split temporally (pre-2020 for training, 2020–2023 for testing, 2024 onward as holdout) to prevent leakage. Advanced feature engineering follows: pricing tiers, genre counts, frequency-encoded tags, and VADER description sentiment scores. The three resulting splits are exported to `data/processed/` for modelling.***

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add the project root to python path to allow importing src
script_path = Path.cwd()
project_root = script_path.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data_cleaning import (
    load_data,
    calculate_success_score,
    filter_zero_reviews,
    transform_features,
    encode_genres,
    split_data,
    add_temporal_features,
    add_pricing_tiers,
    add_genre_count,
    fit_tag_frequencies,
    add_tag_frequency_features,
    add_description_sentiment
)

### Transition to Modular Source Code (`src/`)

With the Exploratory Data Analysis (EDA) phase complete, the data cleaning requirements and feature transformations have stabilized, meaning few (if any) further design changes are expected. 

To prepare for downstream modeling and avoid duplicate code, we transition core preprocessing logic from ad-hoc notebook cells into a modular, production-ready source code directory (`src/`). All preprocessing operations are defined in [`src/data_cleaning.py`](file:///Users/banananakun./Documents/project/Predicting-Video-Game-Commercial-Success-on-Steam/src/data_cleaning.py). This guarantees:
- **Consistency:** The same code handles train, test, and validation splits.
- **Maintainability:** Preprocessing updates can be version-controlled and tested in a single file.
- **Reusability:** Future modeling/inference notebooks can import the pipeline directly.

In [2]:
data_path = project_root / "data" / "processed" / "games_clean_r1.csv"
print(f"Loading data from: {data_path}")
df = load_data(data_path)
print(f"Loaded {len(df)} rows.")

Loading data from: /Users/banananakun./Documents/project/Predicting-Video-Game-Commercial-Success-on-Steam/data/processed/games_clean_r1.csv


Loaded 113065 rows.


## Target Variable & Zero-Review Filter

***The Wilson confidence interval lower bound is computed here via a vectorized implementation, operating directly on the `Positive` and `Negative` arrays rather than row-wise, avoiding the overhead of `apply`. From this, the target variable `success_score = log1p(Review_Count) × wilson_lb` is constructed.***

***Games with a `Review_Count` of zero are then excluded. With no reviews, the Wilson lower bound collapses to 0.0 and `success_score` is identically zero regardless of any other feature, these entries carry no learnable signal for the target and are dropped before modelling.***

In [3]:
df = calculate_success_score(df)
print("Target variable calculated. Summary statistics:")
print(df[['Positive', 'Negative', 'Review_Count', 'wilson_lb', 'success_score']].describe())

df_filtered = filter_zero_reviews(df)
print(f"\nFiltered out zero-review games. Rows remaining: {len(df_filtered)}")

Target variable calculated. Summary statistics:
           Positive      Negative  Review_Count      wilson_lb  success_score
count  1.130650e+05  1.130650e+05  1.130650e+05  113065.000000  113065.000000
mean   1.121464e+03  1.828297e+02  1.304294e+03       0.391616       1.700395
std    2.913343e+04  5.596425e+03  3.387854e+04       0.327573       2.070077
min    0.000000e+00  0.000000e+00  0.000000e+00       0.000000       0.000000
25%    0.000000e+00  0.000000e+00  0.000000e+00       0.000000       0.000000
50%    7.000000e+00  1.000000e+00  9.000000e+00       0.409275       0.951432
75%    4.400000e+01  1.200000e+01  5.800000e+01       0.689961       2.673162
max    7.642084e+06  1.173003e+06  8.815087e+06       0.994553      13.860372

Filtered out zero-review games. Rows remaining: 81837


**Observation:**
- Filtering out zero-review games drops approximately 31k rows (~28% of the cleaned dataset). This is expected since many Steam releases have minimal traction and do not receive any user reviews.
- The maximum `success_score` of 13.86 is consistent with the top-performing games identified during the final exploratory data analysis (Notebook 3).

## Transform Features

***Transformations follow directly from the feature shortlist established in notebook 03. Features with heavy right skew: `game_age_days`, `Price`, `Achievements`, and `DLC count` receive `log1p` transforms; `is_free` is derived as a binary flag from `Price == 0` rather than folding free-to-play titles into the same continuous distribution as paid games; `num_tags` is carried as-is; and `primary_genre` is extracted as the first entry in the `Genres` string, deferred to the next section for one-hot encoding. All logic is implemented in `src/data_cleaning.py` and called here via `transform_features`.***

| Feature | Transform | Notes |
|---|---|---|
| `success_score` | target variable | `log1p(review_count) × wilson_lb`, defined and justified in notebook 03 |
| `num_tags` | as-is (int) | strongest individual correlation (r ≈ 0.52); proxy for discoverability and tagging effort |
| `game_age_days` | log1p | second strongest correlation (r ≈ 0.41); confirmed confounder, carried as a feature rather than adjusted for in the target |
| `Price` | log1p | right-skewed; weak standalone correlation (r ≈ 0.03); free-to-play flagged separately |
| `is_free` | binary flag | derived from `Price == 0`, free and paid titles likely follow different dynamics |
| `primary_genre` | one-hot (top 20) | first entry in `Genres`; rare genres showed unstable small sample averages and are grouped as `Other` |
| `Achievements` | log1p | weak standalone correlation (r ≈ 0.08), but median score shifted meaningfully across achievement tiers in EDA |
| `DLC count` | log1p | weakest correlation (r ≈ 0.06); no visible trend in scatter, but right-skewed distribution still warrants the transform |

In [4]:
df_transformed = transform_features(df_filtered, snapshot_date='2026-01-05')
print(f"Features transformed. Final shape: {df_transformed.shape}")
print(df_transformed[['game_age_days', 'log_game_age_days', 'log_price', 'is_free', 'num_tags', 'primary_genre', 'log_achievements', 'log_dlc_count']].head())

Features transformed. Final shape: (81837, 46)
   game_age_days  log_game_age_days  log_price  is_free  num_tags  \
0           3447           8.145550   1.830980        0         4   
1           2436           7.798523   1.790091        0        16   
5            272           5.609472   3.610648        0        18   
7           1704           7.441320   0.688135        0        12   
8           1484           7.303170   0.463734        0         7   

  primary_genre  log_achievements  log_dlc_count  
0     Adventure          0.000000       0.000000  
1        Casual          0.000000       0.000000  
5    Simulation          0.000000       0.693147  
7        Casual          2.302585       0.000000  
8        Action          4.615121       0.693147  


## Temporal Split

***The dataset is divided into three non-overlapping splits by release year. Games released before 2020 form the training set; 2020–2023 form the test set; 2024 onward serve as an out-of-time validation holdout. This ordering ensures no future release information leaks into the training distribution, a random split would allow the model to learn from games it would never have observed at the point of prediction. The 2024+ cohort is held out entirely until final evaluation and is not used for any model selection or hyperparameter decisions.***

***As noted in notebook 03, older games carry a structural advantage in `success_score` through time-on-market alone. `game_age_days` is included as a feature to give the model visibility into this effect, though it cannot fully resolve it, a 2010 release and a 2022 release remain not directly comparable on `success_score` alone. This is acknowledged as a known limitation of the project.***

In [5]:
train_df, test_df, val_df = split_data(df_transformed)
print(f"Train set shape: {train_df.shape} (releases before 2020)")
print(f"Test set shape: {test_df.shape} (releases from 2020 to 2023)")
print(f"Validation set shape: {val_df.shape} (releases from 2024 onward)")

Train set shape: (29128, 46) (releases before 2020)
Test set shape: (34531, 46) (releases from 2020 to 2023)
Validation set shape: (18178, 46) (releases from 2024 onward)


## One-Hot Encode Genres

***`primary_genre` is one-hot encoded against the top 20 genres by frequency but that frequency is determined from the training set only. Fitting the genre vocabulary on the full dataset before splitting would allow test and validation distributions to influence which genres are retained, constituting a form of leakage. Genres absent from the top 20 training list are mapped to `Other` in the test and validation sets, and any dummy columns present in those sets but absent from training are dropped to guarantee consistent feature shapes across all three splits.***

In [6]:
train_df, top_genres = encode_genres(train_df, top_n=20)
test_df, _ = encode_genres(test_df, top_genres=top_genres)
val_df, _ = encode_genres(val_df, top_genres=top_genres)

print(f"Top 20 Genres determined from Train set: {top_genres}")
print(f"Train set columns after genre encoding: {train_df.shape[1]}")
print(f"Test set columns after genre encoding: {test_df.shape[1]}")
print(f"Validation set columns after genre encoding: {val_df.shape[1]}")

Top 20 Genres determined from Train set: ['Action', 'Adventure', 'Casual', 'Indie', 'Simulation', 'Strategy', 'RPG', 'Violent', 'Free To Play', 'Racing', 'Sexual Content', 'Sports', 'Education', 'Massively Multiplayer', 'Nudity', 'Unknown', 'Gore', 'Utilities', 'Early Access', 'Design & Illustration']
Train set columns after genre encoding: 67
Test set columns after genre encoding: 67
Validation set columns after genre encoding: 67


## Advanced Feature Engineering

***To support final downstream models, we perform the remaining feature engineering directly across the splits:***
1. ***Temporal: `release_year` derived from release dates.***
2. ***Pricing Tiers: One-hot dummy columns for price brackets: `price_tier_free`, `price_tier_budget` (up to $5), `price_tier_mid` ($5 to $20), and `price_tier_premium` (>$20).***
3. ***Genre count: `num_genres` (number of genres).***
4. ***Tag frequency aggregates: mapping tags to their frequency counts inside the training corpus to represent niche vs broad categories.***
5. ***NLP Description Sentiment: VADER compound score computed on the textual description field.***

***Note: All feature mappings (like tag frequency counts) are fitted exclusively on the training split to prevent any data leakage.***

In [7]:
print("Engineering release_year, pricing tiers, and genre counts...")
train_df = add_temporal_features(train_df)
test_df = add_temporal_features(test_df)
val_df = add_temporal_features(val_df)

train_df = add_pricing_tiers(train_df)
test_df = add_pricing_tiers(test_df)
val_df = add_pricing_tiers(val_df)

train_df = add_genre_count(train_df)
test_df = add_genre_count(test_df)
val_df = add_genre_count(val_df)

print("Fitting training set tag frequencies...")
tag_freq_map = fit_tag_frequencies(train_df)
train_df = add_tag_frequency_features(train_df, tag_freq_map)
test_df = add_tag_frequency_features(test_df, tag_freq_map)
val_df = add_tag_frequency_features(val_df, tag_freq_map)

print("Computing VADER description sentiment score (takes ~75 seconds total)...")
train_df = add_description_sentiment(train_df)
test_df = add_description_sentiment(test_df)
val_df = add_description_sentiment(val_df)

print(f"Final Train shape: {train_df.shape}")
print(f"Final Test shape: {test_df.shape}")
print(f"Final Validation shape: {val_df.shape}")

Engineering release_year, pricing tiers, and genre counts...


Fitting training set tag frequencies...


Computing VADER description sentiment score (takes ~75 seconds total)...


Final Train shape: (29128, 79)
Final Test shape: (34531, 79)
Final Validation shape: (18178, 79)


## Sanity Checks

***Four assertions verify the pipeline before export. Row counts across all three splits must sum to `df_filtered` exactly, any mismatch would indicate rows lost or duplicated during the temporal split. All splits must have identical column counts and column names alignment, confirming that `encode_genres` and `add_pricing_tiers` aligned the dummy columns to the training vocabulary correctly. `success_score` must be non-negative across every split. Finally, there must be no NaN values present in the newly engineered feature columns.***

In [8]:
# 1. Row counts check
assert len(train_df) + len(test_df) + len(val_df) == len(df_filtered), "Row count mismatch after split"

# 2. Column count and name alignment check
assert train_df.shape[1] == test_df.shape[1] == val_df.shape[1], \
    f"Column mismatch: train={train_df.shape[1]}, test={test_df.shape[1]}, val={val_df.shape[1]}"
assert (train_df.columns == test_df.columns).all(), "Train and Test columns do not align"
assert (train_df.columns == val_df.columns).all(), "Train and Validation columns do not align"

# 3. Target value checks
assert (train_df['success_score'] >= 0).all()
assert (test_df['success_score'] >= 0).all()
assert (val_df['success_score'] >= 0).all()

# 4. NaN verification in engineered features
new_cols = [
    'release_year', 'price_tier_free', 'price_tier_budget', 'price_tier_mid', 'price_tier_premium',
    'num_genres', 'tag_freq_mean', 'tag_freq_max', 'tag_freq_min', 'tag_freq_sum', 'desc_sentiment_score'
]
assert not train_df[new_cols].isnull().any().any(), "NaNs found in training features"
assert not test_df[new_cols].isnull().any().any(), "NaNs found in test features"
assert not val_df[new_cols].isnull().any().any(), "NaNs found in validation features"

print(f"Sanity check passed. {len(train_df)} / {len(test_df)} / {len(val_df)} rows across train/test/val.")
print(f"Perfect feature alignment verified: All splits have {train_df.shape[1]} columns.")

Sanity check passed. 29128 / 34531 / 18178 rows across train/test/val.
Perfect feature alignment verified: All splits have 79 columns.


## Export

***With the target variable constructed, zero-review entries removed, features transformed, the temporal split applied, genres encoded, and advanced feature extraction completed, the three final modelling-ready splits are exported to `data/processed/` as `games_train.csv`, `games_test.csv`, and `games_val.csv`.***

In [9]:
train_out = project_root / "data" / "processed" / "games_train.csv"
test_out = project_root / "data" / "processed" / "games_test.csv"
val_out = project_root / "data" / "processed" / "games_val.csv"

train_df.to_csv(train_out, index=False)
test_df.to_csv(test_out, index=False)
val_df.to_csv(val_out, index=False)

print(f"Saved Train set with features to {train_out}")
print(f"Saved Test set with features to {test_out}")
print(f"Saved Validation set with features to {val_out}")

Saved Train set with features to /Users/banananakun./Documents/project/Predicting-Video-Game-Commercial-Success-on-Steam/data/processed/games_train.csv
Saved Test set with features to /Users/banananakun./Documents/project/Predicting-Video-Game-Commercial-Success-on-Steam/data/processed/games_test.csv
Saved Validation set with features to /Users/banananakun./Documents/project/Predicting-Video-Game-Commercial-Success-on-Steam/data/processed/games_val.csv
